# Ablation Study for Reviewer Response

This notebook runs the reviewer-facing ablation study:

1. **Performance ablation**
   - CNN only
   - CNN + LSTM
   - CNN + LSTM + multitask

2. **Explainability ablation**
   - Grad-CAM
   - ROI
   - LLM

The performance ablation reports Binary Accuracy, AUC, and Macro F1. The explainability ablation reports which explanation outputs are added by each component.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 1. Path Configuration

Update these paths if your Drive layout is different.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/Capstone/HUFS.CSE.DE-fake-it"
DATA_ROOT = "/content/drive/MyDrive/Capstone/Dataset/ff++"
METADATA_PATH = "/content/drive/MyDrive/Capstone/Dataset/ff++/Global_metadata.csv"
OUTPUT_DIR = f"{PROJECT_DIR}/model/ablation_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("METADATA_PATH:", METADATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("cwd:", os.getcwd())

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f"PROJECT_DIR not found: {PROJECT_DIR}")
if not os.path.exists(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")
if not os.path.exists(METADATA_PATH):
    raise FileNotFoundError(f"METADATA_PATH not found: {METADATA_PATH}")

## 2. Install Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q opencv-python timm scikit-learn pandas tqdm

## 3. Runtime Check

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Quick Smoke Test

Run this first to verify that paths and scripts work. It trains on a tiny subset, so the numbers are **not** paper results.

In [ ]:
!python model/ablation_study.py   --data-root "$DATA_ROOT"   --metadata "$METADATA_PATH"   --backbone EfficientNet-b0   --sequence-length 20   --batch-size 2   --epochs 1   --num-workers 2   --limit-train 16   --limit-val 8   --feature-chunk-size 16   --gradient-checkpointing   --amp   --log-every 1   --output "$OUTPUT_DIR/performance_ablation_smoke.csv"

In [ ]:
import pandas as pd

smoke_path = f"{OUTPUT_DIR}/performance_ablation_smoke.csv"
pd.read_csv(smoke_path)

## 5. Performance Ablation

Use this for the table in the paper. Adjust `EPOCHS`, `SEQUENCE_LENGTH`, and `BATCH_SIZE` depending on GPU memory and available time.

In [ ]:
EPOCHS = 5
SEQUENCE_LENGTH = 60
BATCH_SIZE = 2
BACKBONE = "EfficientNet-b0"
VARIANTS = "cnn_only,cnn_lstm,cnn_lstm_multitask"

performance_output = f"{OUTPUT_DIR}/performance_ablation.csv"

!python model/ablation_study.py   --data-root "$DATA_ROOT"   --metadata "$METADATA_PATH"   --backbone "$BACKBONE"   --variants "$VARIANTS"   --sequence-length "$SEQUENCE_LENGTH"   --batch-size "$BATCH_SIZE"   --epochs "$EPOCHS"   --num-workers 2   --feature-chunk-size 16   --gradient-checkpointing   --amp   --log-every 10   --output "$performance_output"

In [ ]:
performance_df = pd.read_csv(performance_output)
performance_df

## 6. Explainability Ablation

This table separates detection performance from explanation capability. LLM does not change classification accuracy; it adds textual interpretability.

In [ ]:
explainability_output = f"{OUTPUT_DIR}/explainability_ablation.csv"

!python model/explainability_ablation.py --output "$explainability_output"

In [ ]:
explainability_df = pd.read_csv(explainability_output)
explainability_df

## 7. Reviewer-Facing Combined Table Draft

Use the measured performance rows for CNN variants, then describe Grad-CAM/ROI/LLM as explanation components.

In [ ]:
combined_rows = []

for _, row in performance_df.iterrows():
    name_map = {
        "cnn_only": "CNN only",
        "cnn_lstm": "CNN + LSTM",
        "cnn_lstm_multitask": "CNN + LSTM + multitask",
    }
    combined_rows.append({
        "Variant": name_map.get(row["variant"], row["variant"]),
        "Binary Acc.": row["binary_acc"],
        "AUC": row["auc"],
        "Macro F1": row["macro_f1"],
        "Explanation Output": row["explanation_output"],
    })

# These explanation rows use the full detector's measured classification metrics.
# Replace with your chosen full-detector row if needed.
full_detector = performance_df[performance_df["variant"] == "cnn_lstm_multitask"].iloc[0]
for _, row in explainability_df.iloc[1:].iterrows():
    combined_rows.append({
        "Variant": row["variant"],
        "Binary Acc.": full_detector["binary_acc"],
        "AUC": full_detector["auc"],
        "Macro F1": full_detector["macro_f1"],
        "Explanation Output": row["explanation_output"],
    })

combined_df = pd.DataFrame(combined_rows)
combined_path = f"{OUTPUT_DIR}/combined_ablation_table.csv"
combined_df.to_csv(combined_path, index=False)
combined_df

In [ ]:
print("Saved files:")
for path in [
    f"{OUTPUT_DIR}/performance_ablation_smoke.csv",
    f"{OUTPUT_DIR}/performance_ablation.csv",
    f"{OUTPUT_DIR}/explainability_ablation.csv",
    f"{OUTPUT_DIR}/combined_ablation_table.csv",
]:
    print(path, "exists=", os.path.exists(path))